In [0]:
from pyspark.sql.functions import col, when, concat, lit, round

spark.sql("USE inventory_ai")

# Load base Gold tables
risk_df = spark.table("inventory_ai.gold_stockout_risk")
repl_df = spark.table("inventory_ai.gold_replenishment_recommendations")

# Join and IMMEDIATELY flatten to avoid ambiguity
base_df = (
    risk_df.join(
        repl_df,
        on=["date", "store_id", "product_family"],
        how="inner"
    )
    .select(
        risk_df.date.alias("date"),
        risk_df.store_id.alias("store_id"),
        risk_df.product_family.alias("product_family"),
        risk_df.current_stock.alias("current_stock"),
        risk_df.expected_demand_during_lead_time.alias("expected_demand_during_lead_time"),
        risk_df.stockout_risk_score.alias("stockout_risk_score"),
        risk_df.stockout_risk_level.alias("risk_level"),
        repl_df.recommended_order_qty.alias("recommended_order_qty"),
        repl_df.replenishment_action.alias("replenishment_action")
    )
)

# AI Risk explanation
base_df = base_df.withColumn(
    "ai_risk_explanation",
    when(
        col("risk_level") == "HIGH",
        concat(
            lit("High stock-out risk because expected demand during lead time ("),
            round(col("expected_demand_during_lead_time"), 1),
            lit(" units) exceeds current stock ("),
            col("current_stock"),
            lit(" units).")
        )
    ).when(
        col("risk_level") == "MEDIUM",
        lit("Moderate stock-out risk due to demand and inventory balance.")
    ).otherwise(
        lit("Low stock-out risk as inventory sufficiently covers expected demand.")
    )
)

# AI Replenishment explanation
base_df = base_df.withColumn(
    "ai_replenishment_explanation",
    when(
        col("replenishment_action") == "REORDER",
        concat(
            lit("Reorder recommended. Suggested quantity: "),
            round(col("recommended_order_qty"), 0),
            lit(" units considering lead time and safety stock.")
        )
    ).otherwise(
        lit("No replenishment required. Inventory level is sufficient.")
    )
)

# Confidence level
base_df = base_df.withColumn(
    "confidence_level",
    when(col("stockout_risk_score") >= 0.8, "HIGH")
    .when(col("stockout_risk_score") >= 0.4, "MEDIUM")
    .otherwise("LOW")
)

# Save FINAL AI Insights Gold table
base_df.select(
    "date",
    "store_id",
    "product_family",
    "ai_risk_explanation",
    "ai_replenishment_explanation",
    "confidence_level"
).write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("inventory_ai.gold_ai_insights")


In [0]:
spark.sql("""
SELECT *
FROM inventory_ai.gold_ai_insights
LIMIT 10
""").show()
